In [4]:
import torch 
import matplotlib.pyplot as plt
import random
import numpy as np
from matplotlib.colors import ListedColormap
import dill
from sparse_generalization.envs.box_world.env import BoxWorldEnv
from sparse_generalization.envs.box_world.wrappers import make_env
from minigrid.wrappers import FullyObsWrapper
import gymnasium as gym
import cv2
import matplotlib.pyplot as plt
from torchmetrics.classification.accuracy import BinaryAccuracy
gym.register('BoxWorldEnv-v1', BoxWorldEnv)

%load_ext autoreload
%autoreload 2

ModuleNotFoundError: No module named 'sparse_generalization.envs.box_world.env'

In [ ]:
with open('../results/bern_mha_20seeds_24_Feb_2026__06h27m.pl', 'rb') as file:
    results = dill.load(file)

In [ ]:
A = torch.eye(100) 
num_rand = 1

for _ in range(4):
    I = torch.eye(100) * 4
    indices = torch.randint(low=0, high=100, size=(2*num_rand,)).reshape(2, num_rand)
    print(indices)
    A[indices[0], indices[1]] = 1
    A = A @ I
    
A.sum()

tensor([[26],
        [43]])
tensor([[19],
        [ 5]])
tensor([[59],
        [60]])
tensor([[78],
        [22]])


tensor(25940.)

In [ ]:
with open('../data/box_world/balanced_dist_500.pl', 'rb') as file:
    dataset = dill.load(file)
    file.close()

In [ ]:
x_train = dataset['X_train']
y_train = dataset['Y_train']
x_test_ind = dataset['X_test_ind']
y_test_ind = dataset['Y_test_ind']
x_test_ood = dataset['X_test_ood']
y_test_ood = dataset['Y_test_ood']
edges_train = dataset['edges_train']
edges_test = dataset['edges_test_ood']

In [ ]:
test = edges_train[0:50]
test = test.view(50 * 1 * 4, 2, 2)
test = test[:, :, 1] * 10 + test[:, :, 0] 
test = test.view(50, 4, 2)
A = torch.zeros(50, 100, 100)
item = test[0]
A[0, item[:, 0], item[:, 1]] = 1

In [ ]:
test[0]

tensor([[[16, 54],
         [11, 12],
         [51, 52],
         [27, 26]]])

In [ ]:
edges_train[3]

tensor([[[[6, 7],
          [5, 8]],

         [[4, 6],
          [5, 6]],

         [[1, 7],
          [2, 7]],

         [[8, 5],
          [7, 5]]]])

In [ ]:
def get_edges(obs, residual=False):
    edges = []
    # get goal-lock 
    x_goal, y_goal = torch.where(obs[:, :, 0] == 8)
    goal = torch.stack([x_goal, y_goal], dim=1).squeeze()
    edges.append(((goal[0]+1, goal[1]), (goal[0], goal[1])))
    # for each key check key + 1 in lock then a combo
    xs, ys = torch.where(x_train[0][:, :, 0] == 13)
    locks = torch.stack([xs, ys], dim=1)
    locks = locks[~(locks == torch.tensor([goal[0]+1, goal[1]])).all(dim=1)]
    
    xs, ys = torch.where(x_train[0][:, :, 0] == 12)
    keys = torch.stack([xs, ys], dim=1)
    for lock in locks:
        key = (lock[0]-1, lock[1])
        edges.append(((lock[0], lock[1]), (key)))
        keys = keys[~(keys == torch.tensor([lock[0]-1, lock[1]])).all(dim=1)]
        
    # remaining key is residual or looking from agent
    first_key = keys.squeeze()
    if not residual: 
        xs, ys = torch.where(x_train[0][:, :, 0] == 10)
        agent = torch.stack([xs, ys], dim=1).squeeze()
        edges.append(((agent[0], agent[1]), (first_key[0], first_key[1])))
    
    return edges

In [ ]:
from sparse_generalization.layers.oracle import MultiHeadAttentionOracle

mha = MultiHeadAttentionOracle(embed_size=3, num_heads=3)
x_train = x_train.view(500, 100, 3).float()
mha(x_train, x_train, x_train, edges_train)[1].shape

torch.Size([500, 100, 100])

In [5]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

# ==========================================
# 00. DETERMINISTIC SEEDING
# ==========================================
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # multi-GPU safety
    
    # Force PyTorch operations to be fully deterministic
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    print(f"Global seed locked to: {seed}")

# Execute seeding before any data generation or module instantiation
seed_everything(42)

# Check for GPU acceleration
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}\n")

# ==========================================
# 0. MOCK / WRAPPER FOR EXTRACTING SPARTAN
# ==========================================
try:
    from sparse_generalization.models.spartan import SPARTAN
except ImportError:
    raise ImportError(
        "Please ensure 'sparse_generalization' is installed or present in your execution directory."
    )

class Logger:
    def __init__(self):
        pass
    def log_metrics(self, a, step):
        pass

# ==========================================
# 1. 4D ONE-HOT SYNTHETIC DATA GENERATION
# ==========================================
width, height = 4, 4
num_embeddings = width * height  # 16 unique classes (0 to 15)
num_samples = 5000  

X_list = []
Y_list = []
half_samples = num_samples // 2

def check_row_rule(grid):
    """Helper to verify if a 4x4 grid meets the Class 1 criteria."""
    for row_idx in range(width):
        row = grid[row_idx]
        if (0 in row and 1 in row) or (8 in row and 9 in row):
            return True
    return False

print("Generating Class 0 samples...")
while len(X_list) < half_samples:
    grid = torch.randperm(num_embeddings).float().view(width, height)
    if not check_row_rule(grid):
        X_list.append(grid.long())  # Keep as a 2D 4x4 discrete grid
        Y_list.append(torch.tensor([0.0]))

print("Generating Class 1 samples...")
while len(X_list) < num_samples:
    target_pair = torch.rand(1).item()
    injected_tokens = [0.0, 1.0] if target_pair > 0.5 else [8.0, 9.0]
    
    forbidden = set(injected_tokens)
    remaining_pool = torch.tensor([float(x) for x in range(num_embeddings) if x not in forbidden], dtype=torch.float32)
    background_tokens = remaining_pool[torch.randperm(len(remaining_pool))]
    
    flat_grid = torch.zeros(num_embeddings)
    target_row = torch.randint(0, width, (1,)).item()
    col_indices = torch.randperm(height)[:2]
    
    idx1 = target_row * height + col_indices[0]
    idx2 = target_row * height + col_indices[1]
    
    flat_grid[idx1] = injected_tokens[0]
    flat_grid[idx2] = injected_tokens[1]
    
    bg_idx = 0
    for i in range(num_embeddings):
        if i != idx1 and i != idx2:
            flat_grid[i] = background_tokens[bg_idx]
            bg_idx += 1
            
    X_list.append(flat_grid.view(width, height).long())
    Y_list.append(torch.tensor([1.0]))

# Stack raw categorical grid layouts
X_raw = torch.stack(X_list)  # Shape: (5000, 4, 4)
Y_data = torch.stack(Y_list)

# --- 4D ONE-HOT ENCODING TRANSFORMATION ---
# 1. F.one_hot maps (5000, 4, 4) -> (5000, 4, 4, 16)
# X_one_hot = F.one_hot(X_raw, num_classes=num_embeddings).float()

# Randomly shuffle dataset indices to interleave classes
shuffle_idx = torch.randperm(num_samples)
X_data = X_raw[shuffle_idx].unsqueeze(-1)
Y_data = Y_data[shuffle_idx]

# Generator with fixed seed for data loader shuffling consistency
g = torch.Generator()
g.manual_seed(42)

dataloader = DataLoader(
    TensorDataset(X_data, Y_data), 
    batch_size=256, 
    shuffle=True,
    worker_init_fn=lambda worker_id: np.random.seed(42 + worker_id),
    generator=g
)

print(f"4D Dataset generation completed successfully!")
print(f"X matrix shape: {X_data.shape} (Samples, Channels/Classes, Height, Width) | Y shape: {Y_data.shape}\n")


# ==========================================
# 2. INSTANTIATING THE SPARTAN OBJECT
# ==========================================
logger = Logger()

model = SPARTAN(
    inp_dim=num_embeddings,  # 16 Channels
    out_dim=1,
    model_dim=32,
    alpha=0.1, 
    lr=1e-3, 
    num_layers=1,
    num_heads=1,
    include_sparsity=True,  
    token_pool=False,  
    lagrangian=False,       
    layernorm=True,   
    embedding_inp=True,     # Uses linear layer tracking instead of embedding table
    agg_pool=True, 
    num_embeddings=num_embeddings,
    device=device,
    logger=logger
).to(device)

print("SPARTAN model configured for 4D inputs and initialized.\n")


# ==========================================
# 3. MODEL TRAINING
# ==========================================
epochs = 2000
print(f"Starting training loop execution across {epochs} epochs...")

losses, accs, sparses, _, _, _, _, _, _ = model.fit(dataloader, num_epochs=epochs, testloaders=[])
print(f"Training loop concluded. Model maximum execution paths: {model.max_paths}\n")


# ==========================================
# 4. EXTRACTING INTERNAL ATTENTION STATES
# ==========================================
model.eval()
sample_x, _ = next(iter(dataloader))
sample_x = sample_x[:1].to(device)  # Slice out a single 4D operational sample

with torch.no_grad():
    out, final_path, mask_attns, attns, masks = model(sample_x)

print("--- Structural Shape Configurations ---")
for i in range(len(attns)):
    is_agg = (i == len(attns) - 1 and model.agg_pool)
    name = f"Layer {i+1}" if not is_agg else "Aggregation Layer"
    print(f"{name} -> Attn: {list(attns[i].shape)} | Mask Attn: {list(mask_attns[i].shape)} | Mask: {list(masks[i].shape)}")

print(f"Final Compiled Path Mask Shape Dimensions: {list(final_path.shape)}\n")


# ==========================================
# 5. VISUALIZING SPARSITY MAPS
# ==========================================
num_plots = len(attns)
fig, axes = plt.subplots(num_plots, 3, figsize=(18, 4.0 * num_plots))

if num_plots == 1:
    axes = [axes]

for idx in range(num_plots):
    is_agg = (idx == num_plots - 1 and model.agg_pool)
    layer_name = f"Layer {idx+1}" if not is_agg else "Aggregation Pool"
    
    # Compress batch dimensional representations out of the rendering pipeline
    attn_img = attns[idx][0].squeeze().cpu().numpy()
    mask_attn_img = mask_attns[idx][0].squeeze().cpu().numpy()
    mask_img = masks[idx][0].squeeze().cpu().numpy()
    
    # Enforce safe 2D mappings for single axis vectors
    if attn_img.ndim == 1:
        attn_img = attn_img[None, :]
        mask_attn_img = mask_attn_img[None, :]
        mask_img = mask_img[None, :]

    is_flat_shape = attn_img.shape[0] == 1
    aspect_mode = 'auto' if is_flat_shape else 'equal'
    interp_mode = 'nearest' if is_flat_shape else None

    # Column 1: Raw Attention Map
    ax_attn = axes[idx][0]
    im1 = ax_attn.imshow(attn_img, cmap='viridis', aspect=aspect_mode, interpolation=interp_mode)
    ax_attn.set_title(f"{layer_name}\nAttention Matrix {attn_img.shape}")
    fig.colorbar(im1, ax=ax_attn, fraction=0.046, pad=0.04)
    
    # Column 2: Masked Attention Map
    ax_m_attn = axes[idx][1]
    im2 = ax_m_attn.imshow(mask_attn_img, cmap='plasma', aspect=aspect_mode, interpolation=interp_mode)
    ax_m_attn.set_title(f"{layer_name}\nMasked Attention {mask_attn_img.shape}")
    fig.colorbar(im2, ax=ax_m_attn, fraction=0.046, pad=0.04)
    
    # Column 3: Sparsity Selection Mask
    ax_mask = axes[idx][2]
    im3 = ax_mask.imshow(mask_img, cmap='inferno', aspect=aspect_mode, interpolation=interp_mode)
    ax_mask.set_title(f"{layer_name}\nExtracted Mask {mask_img.shape}")
    fig.colorbar(im3, ax=ax_mask, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()


# ==========================================
# 6. SOFTMAX AXIS AND SUM VERIFICATION
# ==========================================
print("=== Softmax Sum Evaluation (Tolerance = 1e-4) ===")

for idx, attn_tensor in enumerate(attns):
    is_agg = (idx == len(attns) - 1 and model.agg_pool)
    layer_name = f"Layer {idx+1}" if not is_agg else "Aggregation Pool"
    
    matrix = attn_tensor[0].squeeze()
    if matrix.ndim == 1:
        matrix = matrix[None, :]
        
    row_sums = matrix.sum(dim=-1)
    col_sums = matrix.sum(dim=-2)
    
    rows_match = torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-4)
    cols_match = torch.allclose(col_sums, torch.ones_like(col_sums), atol=1e-4)
    
    print(f"\n[{layer_name}] Operational Tensor Target Dimensions: {list(matrix.shape)}")
    print(f"  -> Rows sum to 1.0? {rows_match} (Min: {row_sums.min().item():.4f}, Max: {row_sums.max().item():.4f})")
    print(f"  -> Cols sum to 1.0? {cols_match} (Min: {col_sums.min().item():.4f}, Max: {col_sums.max().item():.4f})")

Global seed locked to: 42
Using device: cpu

Generating Class 0 samples...
Generating Class 1 samples...
4D Dataset generation completed successfully!
X matrix shape: torch.Size([5000, 4, 4, 1]) (Samples, Channels/Classes, Height, Width) | Y shape: torch.Size([5000, 1])

SPARTAN model configured for 4D inputs and initialized.

Starting training loop execution across 2000 epochs...


Epoch: 406:  20%|██        | 406/2000 [02:50<11:09,  2.38it/s, loss=0.000353, acc=1, sparse_loss=0.000134, edges=27.1]   


KeyboardInterrupt: 

In [ ]:
paths = torch.ones((16, 16)) * 1 + torch.eye(16)
test = torch.zeros((1, 16))
test[:, -1] = 1
test[:, -2] = 1
(test @ paths).sum()

tensor(34.)

In [ ]:
paths = torch.ones((16, 16)) * 1 + torch.eye(16)
for l in range(0):
    multiplier = torch.ones((16, 16)) + torch.eye( 16)
    paths = paths @ multiplier

multiplier = torch.ones((1, 16)) * 1
paths = multiplier @ paths
paths.sum()

tensor(272.)